# Notebook 2: Model Training

Trains three object detection models:
1. **YOLOv11n** (Ultralytics) — latest YOLO, speed + accuracy benchmark
2. **YOLOv8n** (Ultralytics) — established YOLO baseline, v8→v11 comparison
3. **SSDLite320-MobileNetV3** (torchvision) — anchor-based mobile model, edge deployment

All models use transfer learning from pre-trained weights (COCO for YOLO, ImageNet backbone for SSDLite).

## Setup


In [ ]:
import csv, sys, time
from pathlib import Path

import torch
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
RUNS_DIR     = PROJECT_ROOT / 'runs'
MODELS_DIR   = PROJECT_ROOT / 'models'
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES     = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
NUM_CLASSES = len(CLASSES) + 1   # +1 for background (torchvision convention)
DATA_YAML   = str(DATA_DIR / 'data.yaml')

# Rewrite data.yaml with the correct absolute path for this machine
def _write_data_yaml():
    content = (
        f"path: {DATA_DIR.as_posix()}\n"
        f"train: images/train\nval: images/val\ntest: images/test\n"
        f"nc: {len(CLASSES)}\nnames: {CLASSES}\n"
    )
    (DATA_DIR / 'data.yaml').write_text(content)

_write_data_yaml()

# ── Hyperparameters (tuned for RTX 1650 — 4 GB VRAM) ──────────────────────
YOLO_EPOCHS  = 100
TORCH_EPOCHS = 30
YOLO_BATCH   = 8     # safe for RTX 1650 at 640px
TORCH_BATCH  = 2     # SSDLite: 2 is safe at 640px input (resized to 320 internally)
LR           = 1e-4
IMG_SIZE     = 640
USE_AMP      = torch.cuda.is_available()

BATCH_SIZE = TORCH_BATCH

print(f'Device  : {DEVICE}')
print(f'AMP     : {USE_AMP}')
print(f'YOLO batch={YOLO_BATCH}  SSDLite batch={TORCH_BATCH}')
print(f'Data    : {DATA_YAML}')

## Shared Dataset Class


In [ ]:
IMAGE_EXTS = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']


def _iter_images(directory: Path):
    """Yields all image files, deduplicated by lowercase filename."""
    seen = set()
    for ext in IMAGE_EXTS:
        for img in directory.glob(ext):
            key = img.name.lower()
            if key not in seen:
                seen.add(key)
                yield img


class DefectDetectionDataset(Dataset):
    """YOLO-format dataset loader compatible with torchvision detection models."""

    def __init__(self, img_dir: Path, lbl_dir: Path, img_size: int = 640):
        self.img_dir  = img_dir
        self.lbl_dir  = lbl_dir
        self.img_size = img_size
        self.images   = sorted(_iter_images(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        lbl_path = self.lbl_dir / (img_path.stem + '.txt')

        img    = Image.open(img_path).convert('RGB').resize((self.img_size, self.img_size))
        tensor = TF.to_tensor(img)      # [C, H, W], values in [0, 1]

        boxes, labels = [], []
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = max(0.0, (cx - bw / 2) * self.img_size)
                y1 = max(0.0, (cy - bh / 2) * self.img_size)
                x2 = min(float(self.img_size), (cx + bw / 2) * self.img_size)
                y2 = min(float(self.img_size), (cy + bh / 2) * self.img_size)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls + 1)  # 0 = background in torchvision

        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
            area     = (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0])
        else:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros(0,      dtype=torch.int64)
            area     = torch.zeros(0,      dtype=torch.float32)

        target = {
            'boxes':    boxes_t,
            'labels':   labels_t,
            'image_id': torch.tensor([idx]),
            'area':     area,
            'iscrowd':  torch.zeros(len(labels_t), dtype=torch.int64),
        }
        return tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_loader(split: str, batch_size: int = BATCH_SIZE, shuffle: bool = False):
    ds = DefectDetectionDataset(
        DATA_DIR / 'images' / split,
        DATA_DIR / 'labels' / split,
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn, num_workers=0)


train_loader = make_loader('train', shuffle=True)
val_loader   = make_loader('val')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## Training helpers


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision


def train_one_epoch(model, optimizer, loader, device, use_amp: bool = USE_AMP):
    model.train()
    scaler     = torch.amp.GradScaler('cuda', enabled=use_amp)
    total_loss = 0.0
    for images, targets in tqdm(loader, desc='  train', leave=False):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            loss_dict = model(images, targets)
            losses    = sum(loss_dict.values())
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += losses.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
    for images, targets in tqdm(loader, desc='  eval ', leave=False):
        images  = [img.to(device) for img in images]
        outputs = model(images)
        preds = [{'boxes':  o['boxes'].cpu(),
                  'scores': o['scores'].cpu(),
                  'labels': o['labels'].cpu()} for o in outputs]
        gts   = [{'boxes':  t['boxes'].cpu(),
                  'labels': t['labels'].cpu()} for t in targets]
        metric.update(preds, gts)
    torch.cuda.empty_cache()
    return metric.compute()


def save_checkpoint(model, optimizer, epoch, metrics, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'epoch':               epoch,
        'model_state_dict':    model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict(),
        'metrics':             metrics,
    }, path)


def log_run(name: str, epochs: int, map50: float, map50_95: float,
            precision: float, recall: float, train_min: float, params_m: float,
            weights: str, notes: str = ''):
    log_path = RUNS_DIR / 'run_log.csv'
    run_id   = f"{name}_{int(time.time())}"
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([
            run_id, name, '', epochs, BATCH_SIZE, IMG_SIZE, LR,
            round(map50,    4), round(map50_95, 4),
            round(precision, 4), round(recall,  4),
            round(train_min, 1), round(params_m, 2), weights, notes
        ])
    print(f'  Logged: {run_id}')


## Section 1: YOLO Training


In [ ]:
from ultralytics import YOLO

# Auto-version: find next unused vN folder
v11_ver = 1
while (RUNS_DIR / 'yolo11n' / f'v{v11_ver}').exists():
    v11_ver += 1

print(f'YOLOv11n → saving to runs/yolo11n/v{v11_ver}')
yolo11 = YOLO('yolo11n.pt')
t0 = time.time()
yolo11.train(
    data     = DATA_YAML,
    epochs   = YOLO_EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = YOLO_BATCH,
    lr0      = LR,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    project  = str(RUNS_DIR / 'yolo11n'),
    name     = f'v{v11_ver}',
    exist_ok = False,
    save     = True,
    amp      = True,
    workers  = 0,        # Windows: must be 0 to avoid multiprocessing errors
    verbose  = True,
)
yolo11_min = (time.time() - t0) / 60

yolo11_val     = yolo11.val(data=DATA_YAML)
yolo11_weights = str(RUNS_DIR / 'yolo11n' / f'v{v11_ver}' / 'weights' / 'best.pt')
yolo11_params  = sum(p.numel() for p in yolo11.model.parameters()) / 1e6

print(f'YOLOv11n  mAP@0.5={yolo11_val.box.map50:.4f}  mAP@0.5:0.95={yolo11_val.box.map:.4f}  ({yolo11_min:.0f} min)')
log_run('YOLOv11n', YOLO_EPOCHS,
        float(yolo11_val.box.map50), float(yolo11_val.box.map),
        float(yolo11_val.box.mp),   float(yolo11_val.box.mr),
        yolo11_min, yolo11_params, yolo11_weights)

## Section 2: YOLOv8n Training

In [ ]:
from ultralytics import YOLO

v8_ver = 1
while (RUNS_DIR / 'yolov8n' / f'v{v8_ver}').exists():
    v8_ver += 1

print(f'YOLOv8n → saving to runs/yolov8n/v{v8_ver}')
yolo8 = YOLO('yolov8n.pt')
t0 = time.time()
yolo8.train(
    data     = DATA_YAML,
    epochs   = YOLO_EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = YOLO_BATCH,
    lr0      = LR,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    project  = str(RUNS_DIR / 'yolov8n'),
    name     = f'v{v8_ver}',
    exist_ok = False,
    save     = True,
    amp      = True,
    workers  = 0,
    verbose  = True,
)
yolo8_min = (time.time() - t0) / 60

yolo8_val     = yolo8.val(data=DATA_YAML)
yolo8_weights = str(RUNS_DIR / 'yolov8n' / f'v{v8_ver}' / 'weights' / 'best.pt')
yolo8_params  = sum(p.numel() for p in yolo8.model.parameters()) / 1e6

print(f'YOLOv8n  mAP@0.5={yolo8_val.box.map50:.4f}  mAP@0.5:0.95={yolo8_val.box.map:.4f}  ({yolo8_min:.0f} min)')
log_run('YOLOv8n', YOLO_EPOCHS,
        float(yolo8_val.box.map50), float(yolo8_val.box.map),
        float(yolo8_val.box.mp),   float(yolo8_val.box.mr),
        yolo8_min, yolo8_params, yolo8_weights)

## Section 3: SSDLite320-MobileNetV3 Training

In [ ]:
from models.ssdlite_detector import build_model as build_ssd

# Auto-version
ssd_ver = 1
while (RUNS_DIR / 'ssdlite' / f'best_v{ssd_ver}.pth').exists():
    ssd_ver += 1

ssd_ckpt_dir = RUNS_DIR / 'ssdlite'
ssd_ckpt_dir.mkdir(parents=True, exist_ok=True)
best_ckpt = ssd_ckpt_dir / f'best_v{ssd_ver}.pth'

ssd = build_ssd(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
ssd_params = sum(p.numel() for p in ssd.parameters()) / 1e6
print(f'SSDLite params: {ssd_params:.1f}M → saving to {best_ckpt.name}')

# Phase 1: freeze backbone (epochs 1-5), train detection head only
for p in ssd.backbone.parameters():
    p.requires_grad = False

ssd_opt   = torch.optim.AdamW([p for p in ssd.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
ssd_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ssd_opt, T_max=5)

best_map50 = 0.0
t0 = time.time()

for epoch in range(1, TORCH_EPOCHS + 1):
    if epoch == 6:
        # Phase 2: unfreeze backbone, lower LR by 10x
        for p in ssd.backbone.parameters():
            p.requires_grad = True
        ssd_opt   = torch.optim.AdamW(ssd.parameters(), lr=LR / 10, weight_decay=1e-4)
        ssd_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ssd_opt, T_max=TORCH_EPOCHS - 5)

    loss    = train_one_epoch(ssd, ssd_opt, train_loader, DEVICE)
    ssd_sched.step()
    metrics = evaluate(ssd, val_loader, DEVICE)
    m50     = float(metrics['map_50'])
    print(f'  Epoch {epoch:3d}/{TORCH_EPOCHS} | loss={loss:.4f} | mAP@0.5={m50:.4f}')
    if m50 > best_map50:
        best_map50 = m50
        save_checkpoint(ssd, ssd_opt, epoch, metrics, best_ckpt)

ssd_train_min = (time.time() - t0) / 60
final_metrics = evaluate(ssd, val_loader, DEVICE)
log_run('SSDLite', TORCH_EPOCHS,
        float(final_metrics['map_50']), float(final_metrics['map']),
        0.0, 0.0, ssd_train_min, ssd_params, str(best_ckpt))

print(f'SSDLite best mAP@0.5: {best_map50:.4f}  ({ssd_train_min:.0f} min)')

## Training Summary


In [ ]:
import pandas as pd

run_df = pd.read_csv(RUNS_DIR / 'run_log.csv')
cols   = ['model', 'epochs', 'map50', 'map50_95', 'precision', 'recall', 'train_time_min', 'params_M']
print(run_df[cols].to_string(index=False))